In [ ]:
!pip install aicsimageio hubmap-ome-utils opencv-python pandas pywavelets torch torchvision

In [17]:
from base64 import b64decode
from io import BytesIO
from pathlib import Path
from pprint import pprint

# Set image path:

In [ ]:
uuids = {{uuids | safe}}

In [ ]:
from ome_utils import find_ome_tiffs


def find_expr_dir(base_dir: Path) -> Path:
    if (d := base_dir / "pipeline_output").is_dir():
        return d / "expr"
    if (d := base_dir / "stitched").is_dir():
        return d / "expressions"
    raise ValueError("Couldn't find image directory")


def find_expr_image(base_dir: Path) -> Path:
    """
    Returns the first OME-TIFF found.
    """
    expr_dir = find_expr_dir(base_dir)
    for ome_tiff in find_ome_tiffs(expr_dir):
        return ome_tiff
    raise ValueError("No OME-TIFFs found")

In [ ]:
image_path = find_expr_image(Path("datasets") / uuids[0])
print(f"{image_path=}")

Deep learning model imports

In [8]:
import aicsimageio
import cv2
import numpy as np
import pandas as pd
import pywt
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF

Decode serialized model from base64 to binary HDF5

In [19]:
model_h5_base64 = b"..."
model_h5_bytesio = BytesIO(b64decode(model_h5_base64))

Define model class and image processing helper function

In [20]:
dest_size = (1024, 1024)


class SimpleCNN(nn.Module):
    def __init__(self, image_dim):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.LeakyReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.LeakyReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        new_width = image_dim[1] // 8
        new_height = image_dim[0] // 8
        self.last_layer_size = new_width * new_height * 128
        self.dropout = nn.Dropout(p=0.3)
        self.fc1 = nn.Linear(self.last_layer_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        out = out.view(-1, self.last_layer_size)
        out = self.dropout(out)
        out = self.fc1(out)
        out = self.sigmoid(out)
        return out


def preprocessImage(im):
    im = cv2.resize(im, dsize=dest_size)

    if np.std(im) != 0:
        im = (im - np.mean(im)) / np.std(im)
        im[im > 3] = 3.0
        im[im < -3] = -3.0
    else:
        im = im - np.mean(im)

    coeffs = pywt.dwt2(im, "db2")
    ll, (lh, hl, hh) = coeffs
    im = np.concatenate(
        (ll[..., None], lh[..., None], hl[..., None], hh[..., None]), axis=2
    )
    im = TF.to_tensor(np.float64(im))
    patches = im.unfold(1, 128, 128).unfold(2, 128, 128)
    patches = patches.transpose(0, 2)
    im = patches.contiguous().view(-1, 4, 128, 128)

    return im

Initialize model and load weights

In [15]:
device = torch.device("cpu")
image_shape = (128, 128)
model = SimpleCNN(image_shape)
model.load_state_dict(torch.load(model_h5_bytesio, map_location=device))
model.eval()

SimpleCNN(
  (conv1): Sequential(
    (0): Conv2d(4, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.01)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.01)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.01)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=32768, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

Score markers

In [ ]:
image = aicsimageio.AICSImage(image_path)
num_markers = len(image.channel_names)
print("Channel names:")
pprint(image.channel_names)

In [ ]:
marker_scores = np.zeros((num_markers, 2))

image_data_squeezed = image.data.squeeze()
for i, channel in enumerate(image.channel_names):
    channel_data = image_data_squeezed[i]
    im = preprocessImage(channel_data)
    with torch.no_grad():
        output = model(im.view(-1, 4, 128, 128).type("torch.FloatTensor"))
    marker_scores[i, 0] += output.max()
    marker_scores[i, 1] += 1

marker_scores[:, 0] /= marker_scores[:, 1]
marker_score_series = pd.Series(
    marker_scores[:, 0],
    index=image.channel_names,
).sort_values(ascending=False)

In [ ]:
marker_score_series